# LANGFENS — RQ1: P-SUP B-Control Official Training on Kaggle Tesla T4x2

Private execution notebook under CC BY-NC 4.0.
Executes strictly on authorized remote GPU hardware (Tesla T4) under CUDA FP16 AMP.
Strict fallback prohibition: Never fall back to local CPU/GPU training.
Capped at exactly 3,600 successful optimizer updates with per-epoch atomic remote mirroring.


In [ ]:
# Phase 1: Hardware & Environment Verification
import subprocess, sys

subprocess.run(["nvidia-smi"], check=True)

import torch
print(f"[GPU] CUDA Available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("FAIL-CLOSED: CUDA GPU is not available on this runtime!")
print(f"[GPU] Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"[GPU {i}]: {torch.cuda.get_device_name(i)}")
assert "Tesla T4" in torch.cuda.get_device_name(0), (
    f"FAIL-CLOSED: Expected Tesla T4 accelerator, got {torch.cuda.get_device_name(0)}"
)

import yaml, transformers, huggingface_hub
print(f"[ENV] PyTorch version: {torch.__version__}")
print(f"[ENV] Transformers version: {transformers.__version__}")
print(f"[ENV] Hugging Face Hub version: {huggingface_hub.__version__}")


In [ ]:
# Phase 2: Credentials & Secrets Resolution (Strict Kaggle Secrets Path)
import os

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token and token.strip():
        os.environ["HF_TOKEN"] = token.strip()
        print("[AUTH] Successfully loaded HF_TOKEN from Kaggle Secrets.")
except Exception as e:
    print(f"[AUTH NOTE] Kaggle Secrets UserSecretsClient query: {e}")

if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"].strip():
    raise RuntimeError(
        "FAIL-CLOSED: HF_TOKEN is required in Kaggle Secrets (or environment) for private remote mirroring. "
        "Attach HF_TOKEN under Add-ons -> Secrets in the Kaggle notebook."
    )
print("[AUTH] Verified HF_TOKEN presence in environment for private remote mirroring.")


In [ ]:
# Phase 3: Clone, Checkout Reviewed Training Commit & Verify Tree
import os, subprocess, sys
from pathlib import Path

repo_dir = Path("/kaggle/working/repo")
if not repo_dir.exists():
    print("[GIT] Cloning repository into /kaggle/working/repo...")
    subprocess.run(
        ["git", "clone", "https://github.com/minkhoaa/Langfens-RQ1-Speech-Research.git", str(repo_dir)],
        check=True
    )

os.chdir(str(repo_dir))
subprocess.run(["git", "fetch", "--all"], cwd=repo_dir, check=True)

PINNED_COMMIT_SHA = "56bf79535e4f697b2eea05594d5d81153e04dbff"
print(f"[GIT] Checking out pinned training commit: {PINNED_COMMIT_SHA}...")
subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT_SHA], cwd=repo_dir, check=True)

actual_head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip().lower()
print(f"[GIT] Verified Training Commit HEAD: {actual_head}")
assert actual_head == PINNED_COMMIT_SHA.lower(), (
    f"FAIL-CLOSED: Commit SHA mismatch: actual {actual_head} != expected {PINNED_COMMIT_SHA}"
)
status = subprocess.check_output(["git", "status", "--porcelain"], cwd=repo_dir, text=True).strip()
assert len(status) == 0, f"Dirty tree: {status}"
print("[GIT] Repository verified clean at pinned training commit.")

sys.path.insert(0, str(repo_dir / "src"))
sys.path.insert(0, str(repo_dir))


In [ ]:
# Phase 4: Mandatory Execution Gate (Fail-Closed Default)
EXECUTE_TRAINING = False

if not EXECUTE_TRAINING:
    print("=" * 80)
    print("[HOLD] Execution gate EXECUTE_TRAINING is False for P-SUP B-Control.")
    print("[HOLD] Training execution is held pending explicit operator authorization.")
    print("[HOLD] Zero optimizer steps, model weight updates, or checkpoints created.")
    print("=" * 80)
    import sys
    sys.exit(0)


In [ ]:
# Phase 5: Official Training Execution (capped at 3,600 successful updates)
import subprocess, sys
from pathlib import Path

staging_input = Path("/kaggle/input/langfens-rq1-fold1-staging")
assert staging_input.is_dir(), f"FAIL-CLOSED: Input staging directory missing: {staging_input}"

local_out_dir = Path("/kaggle/working/checkpoints/RQ1-P-SUP/b_control")
local_out_dir.mkdir(parents=True, exist_ok=True)

print(f"[LAUNCH] Starting Official P-SUP B-Control Training on Tesla T4...")
print(f"[LAUNCH] Run ID: psup_b_control")
print(f"[LAUNCH] Target Remote: minkhoaa/langfens-rq1-cross-fold-checkpoints:psup/b_control/")

# Execution command invoked via entrypoint upon operator authorization
train_cmd = [
    sys.executable,
    "deploy/kaggle/kaggle_entrypoint.py",
    "--arm", "B-Control",
    "--commit-sha", PINNED_COMMIT_SHA,
    "--kaggle-input-dir", str(staging_input),
    "--run-id", "psup_b_control",
    "--remote-mirror-backend", "huggingface",
    "--remote-mirror-target", "minkhoaa/langfens-rq1-cross-fold-checkpoints",
]
print("[LAUNCH] Checked command configured:")
print("  " + " ".join(train_cmd))


In [ ]:
# Phase 6: Post-Training Endpoint Classification and Durability Readback
print("=" * 80)
print("[STATUS] Post-training endpoint classification placeholder for P-SUP B-Control.")
print("=" * 80)
